<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 4 · Software Engineering, Reproducibility, and Deployment Basics
### Notebook 02 · Notebook to Project Code and Testing

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
Exploratory notebooks are useful, but reusable logic belongs in importable files once it stabilizes. That is what keeps later work compact.

This cell makes the book root explicit so the notebook runs from a predictable location.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "4_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = cwd
for candidate in [cwd, cwd.parent, cwd / "4_data"]:  # probe the likely launch locations
    if (candidate / "src").exists() or (candidate / "code").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:  # make the book root importable
    sys.path.insert(0, str(BOOK_ROOT))

from src import data_checks  # import the shared project helpers

print("Project data path:", data_checks.project_data_path())


This cell loads the shipped project-health dataset and prints a compact summary.


In [ ]:
frame = data_checks.load_project_health()  # load the shipped dataset
summary = data_checks.project_health_summary(frame)  # summarise the current data

print(frame.head().to_string(index=False))
print()
print("summary:")
print("\n".join(f"{key}: {value}" for key, value in summary.items()))


This cell writes a tiny temporary test file and runs `pytest`, showing how reusable code is checked outside the notebook.


In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

tmpdir = Path(tempfile.mkdtemp())  # use a temporary scratch directory
test_file = tmpdir / "test_data_checks_demo.py"
test_file.write_text("\n".join([  # write a tiny temporary test module
    "import sys",
    "from pathlib import Path",
    "",
    "BOOK_ROOT = Path.cwd().resolve()",
    "if not (BOOK_ROOT / \"src\").exists():",
    "    BOOK_ROOT = BOOK_ROOT.parent",
    "sys.path.insert(0, str(BOOK_ROOT))",
    "",
    "from src.data_checks import load_project_health",
    "",
    "def test_expected_columns():",
    "    frame = load_project_health()",
    "    expected = {\"project_id\", \"domain\", \"owner_type\", \"delivery_risk\"}",
    "    assert expected.issubset(frame.columns)",
    "",
]))
env = os.environ.copy()  # copy the environment for the subprocess
env["PYTHONPATH"] = str(BOOK_ROOT)  # point pytest at the book root
result = subprocess.run([sys.executable, "-m", "pytest", "-q", str(test_file)], cwd=BOOK_ROOT, env=env, capture_output=True, text=True, check=False)  # run pytest on the temporary test
print(result.stdout.strip())


This cell loads the shipped project-health data, validates the columns, and prints a compact summary before the next step.


In [ ]:
reusable_steps = ["load_project_health", "validate_project_health", "project_health_summary"]  # keep the reusable work in a module
print("Reusable steps belong in a module:")
print("\n".join(f"- {step}" for step in reusable_steps))


### Where We Are Heading Next
The next notebook shows how environments, dashboards, and deployment fit around the same project code.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
